# Software Engineering Laboratory Notebook

This notebook is the cumulative Python record for all Software Engineering Laboratory experiments. Each lab section should contain the aim, execution cells, observed output, and test evidence. Source code, input/output files, reports, and screenshots are kept in the corresponding folder under `labs/`.

In [1]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "labs").exists():
    raise RuntimeError("Run this notebook from the repository root.")

print("Repository root verified.")


Repository root verified.


## Lab 1: Student Grade Processing System

**Experiment:** Student Grade Processing System - Functional Testing

**Aim:** Read the uploaded Excel sheet, calculate total marks, assign grades for valid records, and report invalid records without assigning a grade.

### Grading Policy

| Total Marks | Grade |
|---:|:---:|
| 90-100 | A |
| 75-89 | B |
| 60-74 | C |
| 35-59 | D |
| 0-34 | F |

In [2]:
from labs.lab01_student_grade_processing.src.main import (
    DEFAULT_INPUT_FILE,
    DEFAULT_OUTPUT_FILE,
    process_workbook,
)

summary = process_workbook(DEFAULT_INPUT_FILE, DEFAULT_OUTPUT_FILE)

print(f"Input workbook : {DEFAULT_INPUT_FILE.relative_to(ROOT)}")
print(f"Output workbook: {DEFAULT_OUTPUT_FILE.relative_to(ROOT)}")
print(f"Total records  : {summary.total_records}")
print(f"Grades assigned: {summary.valid_records}")
print(f"Invalid records: {summary.invalid_records}")

if summary.errors:
    print("Validation errors:")
    for error in summary.errors:
        print(f"  Row {error['row']} ({error['roll_number']}): {error['message']}")


Input workbook : labs\lab01_student_grade_processing\data\input\Student_Records_Input.xlsx
Output workbook: labs\lab01_student_grade_processing\data\output\Student_Records_Updated.xlsx
Total records  : 12
Grades assigned: 9
Invalid records: 3
Validation errors:
  Row 8 (2026IT007): End examination marks out of range
  Row 9 (2026ECE008): Missing input
  Row 10 (2026ME009): Invalid data type


In [3]:
from openpyxl import load_workbook

workbook = load_workbook(DEFAULT_OUTPUT_FILE)
sheet = workbook.active
lab_test_rows = [7, 8, 9, 10]

observed_results = []
for row in lab_test_rows:
    marks = [sheet.cell(row, column).value for column in range(5, 9)]
    numeric_marks = [mark for mark in marks if isinstance(mark, (int, float))]
    total = sum(numeric_marks) if len(numeric_marks) == 4 else None
    observed_results.append(
        {
            "Roll Number": sheet.cell(row, 1).value,
            "Total Marks": total,
            "Grade": sheet.cell(row, 9).value,
            "Error Message": sheet.cell(row, 10).value,
        }
    )

observed_results


[{'Roll Number': '2026CS006',
  'Total Marks': 60,
  'Grade': 'C',
  'Error Message': None},
 {'Roll Number': '2026IT007',
  'Total Marks': 101,
  'Grade': None,
  'Error Message': 'End examination marks out of range'},
 {'Roll Number': '2026ECE008',
  'Total Marks': None,
  'Grade': None,
  'Error Message': 'Missing input'},
 {'Roll Number': '2026ME009',
  'Total Marks': None,
  'Grade': None,
  'Error Message': 'Invalid data type'}]

### Functional Test Cases

| Test Case ID | Scenario | Expected Result | Actual Result | Status | Error Message |
|---|---|---|---|---|---|
| TC-01 | Invalid total/range | No grade for invalid marks | Grade blank | PASS | End examination marks out of range |
| TC-02 | Valid student record | Grade C for total marks 60 | Grade C | PASS | Not applicable |
| TC-03 | Missing input | No grade when a mark is blank | Grade blank | PASS | Missing input |
| TC-04 | Invalid data type | No grade when a mark is non-numeric | Grade blank | PASS | Invalid data type |

## Lab 2: Library Management System

**Experiment:** Library Management System – Functional Testing

**Aim:** Develop a Library Management System that reads an Excel spreadsheet, validates book records, determines the current status, and reports invalid records.

### Validation Rule

```
Total Copies = Available Copies + Issued Copies
```

### Status Policy

| Condition | Status |
|:----------|:-------|
| Available Copies > 0 | Available |
| Available Copies = 0 | Issued Out |
| Invalid record | Error |

In [4]:
from labs.lab02_library_management_system.src.main import (
    DEFAULT_INPUT_FILE as LAB2_INPUT,
    DEFAULT_OUTPUT_FILE as LAB2_OUTPUT,
    process_workbook as process_library,
)

summary = process_library(LAB2_INPUT, LAB2_OUTPUT)

print(f"Input workbook : {LAB2_INPUT.relative_to(ROOT)}")
print(f"Output workbook: {LAB2_OUTPUT.relative_to(ROOT)}")
print(f"Total records  : {summary.total_records}")
print(f"Status updated : {summary.valid_records}")
print(f"Invalid records: {summary.invalid_records}")

if summary.errors:
    print("Validation errors:")
    for error in summary.errors:
        print(f"  Row {error['row']} ({error['book_id']}): {error['message']}")


Input workbook : labs\lab02_library_management_system\data\input\Library_Records_Input.xlsx
Output workbook: labs\lab02_library_management_system\data\output\Library_Records_Updated.xlsx
Total records  : 10
Status updated : 10
Invalid records: 0


In [5]:
from openpyxl import load_workbook

workbook = load_workbook(LAB2_OUTPUT)
sheet = workbook.active

observed_results = []
for row in range(2, min(5, sheet.max_row + 1)):
    observed_results.append(
        {
            "Book ID": sheet.cell(row, 1).value,
            "Total Copies": sheet.cell(row, 5).value,
            "Available": sheet.cell(row, 6).value,
            "Issued": sheet.cell(row, 7).value,
            "Status": sheet.cell(row, 8).value,
        }
    )

observed_results


[{'Book ID': 'B001',
  'Total Copies': 10,
  'Available': 6,
  'Issued': 4,
  'Status': 'Available'},
 {'Book ID': 'B002',
  'Total Copies': 8,
  'Available': 3,
  'Issued': 5,
  'Status': 'Available'},
 {'Book ID': 'B003',
  'Total Copies': 5,
  'Available': 0,
  'Issued': 5,
  'Status': 'Issued Out'}]

### Functional Test Cases

| Test Case ID | Scenario | Expected Result | Actual Result | Status | Error Message (displayed) |
|---|---|---|---|---|---|
| TC-01 | Invalid Book Record | Status remains blank | Status blank | PASS | Total copies does not equal available + issued |
| TC-02 | Book Completely Issued | Status is Issued Out | Status = Issued Out | PASS | Not applicable |
| TC-03 | Missing Input | Status remains blank | Status blank | PASS | Missing input |
| TC-04 | Invalid Data Type | Status remains blank | Status blank | PASS | Invalid data type |
| TC-05 | Negative Inventory | Status remains blank | Status blank | PASS | Negative inventory value |

## Later Labs

Add subsequent experiments below this section using the same compact pattern: aim, execution cell, observed result, and test evidence. Keep detailed source and artifacts inside that lab's folder under `labs/`.